# Pydantic AI Sandbox using `pydantic-ai-backend`

https://vstorm-co.github.io/pydantic-ai-backend/

In [1]:
!docker pull python:3.13-slim

3.13-slim: Pulling from library/python
Digest: sha256:5a7bc62572092e849a3de833858cd2e5542fe24bc49e3fc54948c8b0d8ca4b29
Status: Image is up to date for python:3.13-slim
docker.io/library/python:3.13-slim


In [2]:
from dataclasses import dataclass
from pydantic_ai import Agent
from pydantic_ai_backends import DockerSandbox, create_console_toolset, RuntimeConfig

/home/simon/work/caktus/llm-learning/.venv/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.2.0)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


Define a custom runtime for your use case

In [3]:
runtime = RuntimeConfig(
    name="ml-env",
    base_image="python:3.12-slim",
    packages=["pandas", "numpy"],
)

Can also use one of the [built-in runtimes](https://vstorm-co.github.io/pydantic-ai-backend/concepts/docker/#built-in-runtimes)

In [4]:
runtime = "python-datascience"

Create the sandbox. Can also use `image="python:3.12-slim"` instead of setting a runtime

In [5]:
sandbox = DockerSandbox(runtime=runtime)

Container starts lazily on first operation, but we can pre-warm it:

In [6]:
sandbox.start()

Create the agent

In [7]:
@dataclass
class Deps:
    backend: DockerSandbox


toolset = create_console_toolset(require_execute_approval=False)
agent = Agent("openai:gpt-4o", deps_type=Deps, toolsets=[toolset])

In [8]:
async def prompt(question):
    result = await agent.run(question, deps=Deps(backend=sandbox))
    print(result.output)

In [9]:
await prompt("Create a fibonacci.py script and run it with n=10")

The `fibonacci.py` script has been created and executed. The Fibonacci sequence for \( n=10 \) is: \([0, 1, 1, 2, 3, 5, 8, 13, 21, 34]\).


In [10]:
await prompt("List all files in the current directory.")

The current directory contains a single file: `fibonacci.py`, which is 272 bytes in size.


In [11]:
await prompt("Show the contents of fibonacci.py")

Here is the content of `fibonacci.py`:

```python
def fibonacci(n):
    sequence = [0, 1]
    for i in range(2, n):
        next_value = sequence[-1] + sequence[-2]
        sequence.append(next_value)
    return sequence[:n]

if __name__ == '__main__':
    n = 10
    print(f"Fibonacci sequence for n={n}: {fibonacci(n)}")
```

This script defines a function `fibonacci` that generates a list of Fibonacci numbers up to `n`. When run as a script, it calculates and prints the Fibonacci sequence for `n=10`.


In [12]:
await prompt(
    "Create a Python script named analyze_iris_dataset.py that loads the iris dataset with sklearn, analyzes it with pandas, "
    "and prints out the pandas DataFrame. Run the script and show me the output."
)

The script `analyze_iris_dataset.py` was executed successfully, and it prints the entire Iris dataset as a pandas DataFrame including the features and the target column. The DataFrame has 150 rows and 5 columns (sepal length, sepal width, petal length, petal width, and target). Here is a glimpse of the DataFrame:

```
     sepal length (cm)  sepal width (cm)  ...  petal width (cm)  target
0                  5.1               3.5  ...               0.2       0
1                  4.9               3.0  ...               0.2       0
2                  4.7               3.2  ...               0.2       0
3                  4.6               3.1  ...               0.2       0
4                  5.0               3.6  ...               0.2       0
..                 ...               ...  ...               ...     ...
145                6.7               3.0  ...               2.3       2
146                6.3               2.5  ...               1.9       2
147                6.5          

In [13]:
await prompt("List all files in the current directory.")

The current directory contains the following files:

1. `analyze_iris_dataset.py` - 291 bytes
2. `fibonacci.py` - 272 bytes


In [14]:
await prompt("Show the contents of analyze_iris_dataset.py")

Here is the content of the `analyze_iris_dataset.py` file:

```python
import pandas as pd
from sklearn.datasets import load_iris

# Load the Iris dataset
iris = load_iris()

# Convert to pandas DataFrame
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)

# Add target column
iris_df['target'] = iris.target

# Print the DataFrame
print(iris_df)
```


## Container Lifecycle

The container will be removed when you call `sandbox.stop()`, unless if you initialized the sandbox with `auto_remove=False` (default is `True`). It also will be removed if idle for at least 1 hour (configurable using the `idle_timeout` arg during initialization).

Can use [`SessionManager`](https://vstorm-co.github.io/pydantic-ai-backend/concepts/docker/#sessionmanager-for-multi-user) to create separate Docker containers for different users

In [15]:
sandbox.stop()